# Thực hành Seaborn từ cơ bản đến ứng dụng

**Ngôn ngữ:** Python 3  
**Thư viện chính:** `seaborn`, `matplotlib`, `pandas`, `numpy`  
**Định hướng ứng dụng:** Data Science, Business Analytics, Economics

Notebook này được thiết kế theo cấu trúc:
**khái niệm → ví dụ → bài tập nhỏ → ứng dụng → bài tổng hợp → bài tự làm**.

> Các bộ dữ liệu trong notebook được tạo mô phỏng để có thể chạy độc lập, không cần Internet.

## Mục tiêu học tập

Sau bài thực hành, người học có thể:

- hiểu Seaborn xây dựng trên Matplotlib như thế nào;
- sử dụng **axes-level functions** và **figure-level functions**;
- vẽ và diễn giải:
  - `lineplot`,
  - `scatterplot`,
  - `barplot`,
  - `countplot`,
  - `boxplot`,
  - `violinplot`,
  - `histplot`,
  - `kdeplot`,
  - `regplot`,
  - `heatmap`,
  - `pairplot`;
- sử dụng `hue`, `style`, `size`, `col`, `row` để trực quan hóa nhiều chiều;
- tùy biến theme, context, palette và kết hợp Seaborn với Matplotlib;
- xây dựng EDA/dashboard nhỏ cho bài toán Data Science, Business và Economics;
- viết nhận xét dựa trên bằng chứng trực quan.


## 0. Chuẩn bị môi trường

Nếu máy chưa có Seaborn:

```bash
pip install seaborn
```

hoặc:

```bash
conda install seaborn
```

Sau đó import các thư viện cần dùng.

### Hàm `sns.set_theme()`

Seaborn có thể thiết lập giao diện mặc định cho toàn bộ notebook bằng:

```python
sns.set_theme(style="whitegrid", context="notebook")
```

Trong đó:

- `style="whitegrid"`: nền trắng có lưới, phù hợp với biểu đồ phân tích dữ liệu.
- `context="notebook"`: cỡ chữ và các thành phần được điều chỉnh phù hợp với Jupyter Notebook.

Một số `style` thường gặp:

```text
whitegrid, darkgrid, white, dark, ticks
```

Một số `context` thường gặp:

```text
paper, notebook, talk, poster
```

> Ta gọi `sns.set_theme()` một lần ở đầu notebook để các biểu đồ phía sau có giao diện thống nhất.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)

sns.set_theme(style="whitegrid", context="notebook")


## 1. Tạo các bộ dữ liệu mẫu

Để tập trung vào **Seaborn và tư duy trực quan hóa**, phần này không sinh dữ liệu
bằng các phân phối ngẫu nhiên phức tạp.

Thay vào đó, ta tạo sẵn ba dataset nhỏ bằng **`list`/`dict` → `pd.DataFrame`**:

1. **Retail** — doanh thu, lợi nhuận, khu vực và nhóm sản phẩm.
2. **Customer** — thông tin khách hàng và trạng thái churn.
3. **Economics** — inflation, unemployment và GDP growth theo quý.

Cách này có ba ưu điểm:

- sinh viên nhìn trực tiếp được dữ liệu đang dùng;
- kết quả luôn giống nhau mỗi lần chạy notebook;
- không bị phân tán bởi code mô phỏng dữ liệu trước khi học Seaborn.

### `pd.DataFrame()`

`pd.DataFrame()` chuyển dữ liệu dạng bảng thành cấu trúc DataFrame của Pandas:

```python
df = pd.DataFrame(data)
```

Trong đó `data` có thể là:

- list các dictionary;
- dictionary các list;
- list các list kèm tên cột.

Ví dụ:

```python
data = [
    {"Region": "North", "Revenue": 450},
    {"Region": "South", "Revenue": 520},
]

df = pd.DataFrame(data)
```

### Lưu DataFrame thành CSV

Nếu muốn dùng lại dataset ở notebook khác:

```python
df.to_csv("sample.csv", index=False)
```

- `index=False` giúp không ghi cột chỉ số `0, 1, 2, ...` vào file CSV.

Trong notebook này, các lệnh lưu CSV được để ở dạng **tùy chọn**.


In [ ]:

# ============================================================
# DATASET 1: RETAIL
# Mỗi dòng là một quan sát bán hàng theo:
# Month × Region × Category
# ============================================================

retail_data = [
    # 2025-01
    {"Month": "2025-01-01", "Region": "North",   "Category": "Electronics", "Revenue": 520, "Profit": 72,  "Orders": 1080},
    {"Month": "2025-01-01", "Region": "Central", "Category": "Home",        "Revenue": 350, "Profit": 70,  "Orders": 760},
    {"Month": "2025-01-01", "Region": "South",   "Category": "Fashion",     "Revenue": 390, "Profit": 103, "Orders": 850},
    {"Month": "2025-01-01", "Region": "Online",  "Category": "Electronics", "Revenue": 610, "Profit": 82,  "Orders": 1340},

    # 2025-02
    {"Month": "2025-02-01", "Region": "North",   "Category": "Home",        "Revenue": 430, "Profit": 86,  "Orders": 930},
    {"Month": "2025-02-01", "Region": "Central", "Category": "Fashion",     "Revenue": 330, "Profit": 88,  "Orders": 720},
    {"Month": "2025-02-01", "Region": "South",   "Category": "Electronics", "Revenue": 560, "Profit": 76,  "Orders": 1180},
    {"Month": "2025-02-01", "Region": "Online",  "Category": "Home",        "Revenue": 540, "Profit": 108, "Orders": 1210},

    # 2025-03
    {"Month": "2025-03-01", "Region": "North",   "Category": "Fashion",     "Revenue": 410, "Profit": 108, "Orders": 900},
    {"Month": "2025-03-01", "Region": "Central", "Category": "Electronics", "Revenue": 470, "Profit": 63,  "Orders": 980},
    {"Month": "2025-03-01", "Region": "South",   "Category": "Home",        "Revenue": 500, "Profit": 100, "Orders": 1080},
    {"Month": "2025-03-01", "Region": "Online",  "Category": "Fashion",     "Revenue": 520, "Profit": 138, "Orders": 1160},

    # 2025-04
    {"Month": "2025-04-01", "Region": "North",   "Category": "Electronics", "Revenue": 590, "Profit": 80,  "Orders": 1230},
    {"Month": "2025-04-01", "Region": "Central", "Category": "Home",        "Revenue": 390, "Profit": 78,  "Orders": 820},
    {"Month": "2025-04-01", "Region": "South",   "Category": "Fashion",     "Revenue": 460, "Profit": 123, "Orders": 990},
    {"Month": "2025-04-01", "Region": "Online",  "Category": "Electronics", "Revenue": 690, "Profit": 93,  "Orders": 1490},

    # 2025-05
    {"Month": "2025-05-01", "Region": "North",   "Category": "Home",        "Revenue": 480, "Profit": 96,  "Orders": 1010},
    {"Month": "2025-05-01", "Region": "Central", "Category": "Fashion",     "Revenue": 370, "Profit": 99,  "Orders": 790},
    {"Month": "2025-05-01", "Region": "South",   "Category": "Electronics", "Revenue": 620, "Profit": 84,  "Orders": 1300},
    {"Month": "2025-05-01", "Region": "Online",  "Category": "Home",        "Revenue": 610, "Profit": 122, "Orders": 1360},

    # 2025-06
    {"Month": "2025-06-01", "Region": "North",   "Category": "Fashion",     "Revenue": 450, "Profit": 120, "Orders": 970},
    {"Month": "2025-06-01", "Region": "Central", "Category": "Electronics", "Revenue": 510, "Profit": 69,  "Orders": 1060},
    {"Month": "2025-06-01", "Region": "South",   "Category": "Home",        "Revenue": 550, "Profit": 110, "Orders": 1170},
    {"Month": "2025-06-01", "Region": "Online",  "Category": "Fashion",     "Revenue": 590, "Profit": 158, "Orders": 1290},
]

retail = pd.DataFrame(retail_data)

# Chuyển cột Month từ chuỗi sang kiểu ngày tháng
retail["Month"] = pd.to_datetime(retail["Month"])


# ============================================================
# DATASET 2: CUSTOMER CHURN
# Mỗi dòng là một khách hàng.
# ============================================================

customer_data = [
    {"Tenure": 3,  "MonthlyCharge": 92, "SupportCalls": 4, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 6,  "MonthlyCharge": 84, "SupportCalls": 3, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 9,  "MonthlyCharge": 78, "SupportCalls": 4, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 12, "MonthlyCharge": 72, "SupportCalls": 2, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 15, "MonthlyCharge": 70, "SupportCalls": 3, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 18, "MonthlyCharge": 68, "SupportCalls": 2, "Contract": "Monthly",  "Churn": 0},
    {"Tenure": 22, "MonthlyCharge": 64, "SupportCalls": 1, "Contract": "Monthly",  "Churn": 0},
    {"Tenure": 27, "MonthlyCharge": 61, "SupportCalls": 1, "Contract": "Monthly",  "Churn": 0},
    {"Tenure": 31, "MonthlyCharge": 58, "SupportCalls": 0, "Contract": "Monthly",  "Churn": 0},
    {"Tenure": 36, "MonthlyCharge": 55, "SupportCalls": 1, "Contract": "Monthly",  "Churn": 0},
    {"Tenure": 5,  "MonthlyCharge": 80, "SupportCalls": 3, "Contract": "Monthly",  "Churn": 1},
    {"Tenure": 25, "MonthlyCharge": 66, "SupportCalls": 2, "Contract": "Monthly",  "Churn": 0},

    {"Tenure": 8,  "MonthlyCharge": 82, "SupportCalls": 3, "Contract": "Annual",   "Churn": 1},
    {"Tenure": 12, "MonthlyCharge": 76, "SupportCalls": 2, "Contract": "Annual",   "Churn": 1},
    {"Tenure": 16, "MonthlyCharge": 74, "SupportCalls": 3, "Contract": "Annual",   "Churn": 1},
    {"Tenure": 20, "MonthlyCharge": 69, "SupportCalls": 2, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 24, "MonthlyCharge": 67, "SupportCalls": 1, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 29, "MonthlyCharge": 63, "SupportCalls": 1, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 34, "MonthlyCharge": 60, "SupportCalls": 0, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 40, "MonthlyCharge": 59, "SupportCalls": 1, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 45, "MonthlyCharge": 56, "SupportCalls": 0, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 50, "MonthlyCharge": 54, "SupportCalls": 1, "Contract": "Annual",   "Churn": 0},
    {"Tenure": 14, "MonthlyCharge": 79, "SupportCalls": 3, "Contract": "Annual",   "Churn": 1},
    {"Tenure": 32, "MonthlyCharge": 62, "SupportCalls": 1, "Contract": "Annual",   "Churn": 0},

    {"Tenure": 10, "MonthlyCharge": 77, "SupportCalls": 3, "Contract": "Two-year", "Churn": 1},
    {"Tenure": 18, "MonthlyCharge": 71, "SupportCalls": 2, "Contract": "Two-year", "Churn": 1},
    {"Tenure": 24, "MonthlyCharge": 66, "SupportCalls": 1, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 30, "MonthlyCharge": 64, "SupportCalls": 1, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 36, "MonthlyCharge": 61, "SupportCalls": 1, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 42, "MonthlyCharge": 58, "SupportCalls": 0, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 48, "MonthlyCharge": 57, "SupportCalls": 0, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 54, "MonthlyCharge": 55, "SupportCalls": 1, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 60, "MonthlyCharge": 52, "SupportCalls": 0, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 66, "MonthlyCharge": 50, "SupportCalls": 0, "Contract": "Two-year", "Churn": 0},
    {"Tenure": 20, "MonthlyCharge": 73, "SupportCalls": 2, "Contract": "Two-year", "Churn": 1},
    {"Tenure": 44, "MonthlyCharge": 59, "SupportCalls": 1, "Contract": "Two-year", "Churn": 0},
]

customers = pd.DataFrame(customer_data)


# ============================================================
# DATASET 3: ECONOMICS
# Mỗi dòng là một quý.
# ============================================================

economy_data = {
    "Quarter": [
        "2023Q1", "2023Q2", "2023Q3", "2023Q4",
        "2024Q1", "2024Q2", "2024Q3", "2024Q4",
        "2025Q1", "2025Q2", "2025Q3", "2025Q4"
    ],
    "Inflation": [
        3.2, 3.8, 4.6, 5.4,
        5.8, 5.2, 4.4, 3.7,
        3.1, 2.8, 2.6, 2.5
    ],
    "Unemployment": [
        5.1, 4.9, 4.7, 4.5,
        4.4, 4.3, 4.5, 4.9,
        5.3, 5.1, 4.8, 4.6
    ],
    "GDP_Growth": [
        4.8, 5.0, 5.2, 4.9,
        4.5, 4.1, 3.2, 1.4,
        0.8, 1.9, 3.1, 4.0
    ],
}

economy = pd.DataFrame(economy_data)


# ============================================================
# Tùy chọn: lưu các dataset để dùng lại dưới dạng CSV
# ============================================================

# retail.to_csv("retail_sample.csv", index=False)
# customers.to_csv("customers_sample.csv", index=False)
# economy.to_csv("economy_sample.csv", index=False)

print("Retail shape:", retail.shape)
print("Customers shape:", customers.shape)
print("Economy shape:", economy.shape)



### Kiểm tra nhanh dữ liệu

Trước khi vẽ, nên xem một vài dòng đầu:

```python
df.head()
```

và kiểm tra missing values:

```python
df.isna().sum()
```

- `head()` giúp kiểm tra cấu trúc cột và giá trị.
- `isna()` trả về `True/False` cho vị trí bị thiếu.
- `sum()` đếm số giá trị thiếu.

Ta kiểm tra cả ba DataFrame trước khi bắt đầu trực quan hóa.


In [ ]:
display(retail.head())
display(customers.head())
display(economy.head())

print("\nMissing values:")
print("Retail:", retail.isna().sum().sum())
print("Customers:", customers.isna().sum().sum())
print("Economy:", economy.isna().sum().sum())


## 2. Seaborn và Matplotlib

Seaborn **không thay thế Matplotlib**.

Có thể hiểu:

- **Matplotlib** tạo `Figure`, `Axes`, tiêu đề, nhãn trục, annotation và layout.
- **Seaborn** cung cấp các hàm trực quan hóa thống kê thuận tiện khi làm việc với DataFrame.

### `plt.subplots()`

Tạo vùng vẽ:

```python
fig, ax = plt.subplots(figsize=(8, 5))
```

- `fig`: toàn bộ Figure.
- `ax`: vùng Axes mà ta sẽ vẽ lên.
- `figsize=(8, 5)`: kích thước hình theo inch.

### `sns.scatterplot()`

Vẽ scatter plot từ DataFrame:

```python
sns.scatterplot(
    data=df,
    x="Revenue",
    y="Profit",
    ax=ax
)
```

Các tham số thường dùng:

- `data=`: DataFrame.
- `x=`, `y=`: tên cột cho hai trục.
- `hue=`: phân nhóm bằng màu.
- `size=`: phân nhóm bằng kích thước điểm.
- `style=`: phân nhóm bằng kiểu marker.
- `alpha=`: độ trong suốt.
- `ax=`: Axes của Matplotlib để vẽ lên.

### `sns.color_palette()`

Lấy bảng màu Seaborn:

```python
sns.color_palette("deep")[0]
```

Ở đây `[0]` lấy màu đầu tiên trong palette `"deep"`.

### Các hàm Matplotlib sau khi vẽ

```python
ax.set_title("...")
ax.set_xlabel("...")
ax.set_ylabel("...")
plt.tight_layout()
plt.show()
```

Seaborn tạo biểu đồ; Matplotlib thường được dùng để hoàn thiện tiêu đề, nhãn và bố cục.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    color=sns.color_palette("deep")[0],
    ax=ax,
)

ax.set_title("Seaborn plot inside a Matplotlib Axes")
ax.set_xlabel("Revenue")
ax.set_ylabel("Profit")

plt.tight_layout()
plt.show()

### Bài tập 1 — Seaborn + Matplotlib

Hãy sửa biểu đồ trên để:

1. phân nhóm theo `Category` bằng `hue`;
2. tăng kích thước Figure;
3. đổi title thành `"Revenue vs Profit by Product Category"`;
4. đặt legend bên ngoài nếu cần.

**Gợi ý:**

```python
sns.scatterplot(..., hue="Category", ax=ax)
ax.legend(...)
```

In [ ]:
# TODO - Bài tập 1

fig, ax = plt.subplots(figsize=(9, 5))

# sns.scatterplot(
#     data=retail,
#     x="Revenue",
#     y="Profit",
#     hue=...,
#     ax=ax,
# )

# ax.set_title(...)
# ax.legend(...)

plt.tight_layout()
plt.show()

## 3. Relational plots — quan hệ giữa các biến

Hai hàm quan trọng:

- `sns.scatterplot()` — quan hệ giữa hai biến số;
- `sns.lineplot()` — xu hướng theo biến có thứ tự hoặc thời gian.

Các tham số mạnh của Seaborn:

- `hue=` → màu;
- `style=` → kiểu marker/line;
- `size=` → kích thước;
- `data=` → DataFrame.


### 3.1 `sns.scatterplot()` — quan hệ giữa hai biến

Scatter plot phù hợp khi muốn xem **mối quan hệ giữa hai biến số**.

Cú pháp cơ bản:

```python
sns.scatterplot(
    data=df,
    x="x_column",
    y="y_column"
)
```

Ví dụ mở rộng:

```python
sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    size="Orders",
    sizes=(30, 250),
    alpha=0.7,
    ax=ax
)
```

Giải thích:

- `hue="Category"`: mỗi nhóm sản phẩm có màu khác nhau.
- `size="Orders"`: số đơn hàng quyết định kích thước điểm.
- `sizes=(30, 250)`: giới hạn kích thước nhỏ nhất/lớn nhất.
- `alpha=0.7`: làm điểm hơi trong suốt để giảm che lấp.

**Câu hỏi phân tích:** Revenue cao hơn có thường đi kèm Profit cao hơn không?


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    size="Orders",
    sizes=(30, 250),
    alpha=0.7,
    ax=ax,
)

ax.set_title("Revenue vs Profit")
plt.tight_layout()
plt.show()

### Diễn giải

Biểu đồ này mã hóa nhiều chiều:

- X: Revenue
- Y: Profit
- màu: Category
- kích thước: Orders

Đây là một ví dụ về **multidimensional visualization**.


### 3.2 `sns.lineplot()` — xu hướng theo thời gian

Line plot phù hợp khi trục X có **thứ tự**, đặc biệt là thời gian.

Cú pháp:

```python
sns.lineplot(
    data=df,
    x="Time",
    y="Value",
    hue="Group",
    marker="o",
    ax=ax
)
```

Trong ví dụ dưới đây:

- `x="Month"`: trục thời gian.
- `y="Revenue"`: biến cần theo dõi.
- `hue="Region"`: mỗi khu vực là một đường.
- `marker="o"`: đánh dấu từng mốc thời gian.

Trước khi vẽ, ta dùng:

```python
retail.groupby(["Month", "Region"], as_index=False)["Revenue"].sum()
```

để cộng Revenue của các Category trong cùng `Month × Region`.

> `lineplot()` cũng có khả năng tự tổng hợp khi có nhiều quan sát cùng X, nhưng tổng hợp trước giúp người học kiểm soát rõ dữ liệu đang được vẽ.


In [ ]:
monthly_region = (
    retail.groupby(["Month", "Region"], as_index=False)["Revenue"]
    .sum()
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.lineplot(
    data=monthly_region,
    x="Month",
    y="Revenue",
    hue="Region",
    marker="o",
    ax=ax,
)

ax.set_title("Monthly Revenue by Region")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 2 — Relational plots

**A. Scatterplot**

Vẽ `Revenue` vs `Profit`:

- `hue="Region"`
- `style="Category"`
- `size="Orders"`

**B. Lineplot**

Vẽ tổng `Profit` theo `Month`, phân nhóm theo `Category`.

**Câu hỏi**

- Region nào xuất hiện nhiều ở vùng Revenue cao?
- Category nào có profit margin có vẻ cao hơn?
- Nhóm sản phẩm nào có xu hướng profit tăng rõ?

In [ ]:
# TODO - Bài tập 2A

fig, ax = plt.subplots(figsize=(10, 6))

# sns.scatterplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 2B

monthly_category_profit = (
    retail.groupby(["Month", "Category"], as_index=False)["Profit"]
    .sum()
)

fig, ax = plt.subplots(figsize=(11, 5))

# sns.lineplot(...)

plt.tight_layout()
plt.show()

## 4. Categorical plots — so sánh giữa các nhóm

Các hàm phổ biến:

- `barplot()` — ước lượng trung tâm theo nhóm;
- `countplot()` — số lượng quan sát;
- `boxplot()` — median, quartiles, spread, outliers;
- `violinplot()` — hình dạng phân phối theo nhóm;
- `stripplot()` / `swarmplot()` — hiển thị từng quan sát.

> Lưu ý: `barplot()` không đơn giản chỉ là vẽ tổng. Mặc định nó tính một estimator (thường là mean).


### 4.1 `sns.barplot()` — so sánh giá trị trung bình giữa các nhóm

`barplot()` thường dùng để so sánh một biến số giữa các nhóm categorical.

Cú pháp:

```python
sns.barplot(
    data=df,
    x="Group",
    y="Value",
    errorbar=None,
    ax=ax
)
```

Mặc định, Seaborn dùng **mean** làm estimator.

Trong ví dụ:

```python
sns.barplot(data=retail, x="Region", y="Revenue", ...)
```

mỗi cột thể hiện **Revenue trung bình** của một Region.

Các tham số:

- `x=`: biến phân nhóm.
- `y=`: biến số cần tổng hợp.
- `errorbar=None`: không hiển thị khoảng sai số.
- `color=`: dùng một màu chung.

> Nếu muốn vẽ **tổng Revenue**, nên `groupby(...).sum()` trước rồi mới vẽ.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=retail,
    x="Region",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=ax,
)

ax.set_title("Average Revenue by Region")
plt.tight_layout()
plt.show()


### 4.2 `sns.countplot()` — đếm số quan sát theo nhóm

`countplot()` trả lời câu hỏi:

> **Mỗi nhóm có bao nhiêu quan sát?**

Cú pháp:

```python
sns.countplot(
    data=df,
    x="Category",
    hue="Subgroup",
    ax=ax
)
```

Trong ví dụ:

- `x="Contract"`: đếm khách hàng theo loại hợp đồng.
- `hue="Churn"`: chia mỗi nhóm theo `Churn = 0/1`.

Khác với `barplot()`, `countplot()` **không cần biến Y** vì nó tự đếm số dòng.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.countplot(
    data=customers,
    x="Contract",
    hue="Churn",
    ax=ax,
)

ax.set_title("Customer Count by Contract and Churn")
plt.tight_layout()
plt.show()


### 4.3 `sns.boxplot()` — so sánh phân phối giữa các nhóm

Boxplot tóm tắt phân phối thông qua:

- median;
- quartiles;
- độ phân tán;
- các điểm có thể là outlier.

Cú pháp:

```python
sns.boxplot(
    data=df,
    x="Group",
    y="NumericValue",
    hue="Subgroup",
    ax=ax
)
```

Trong ví dụ:

- `x="Contract"`: loại hợp đồng.
- `y="MonthlyCharge"`: phí hàng tháng.
- `hue="Churn"`: so sánh khách churn và không churn trong từng contract.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=customers,
    x="Contract",
    y="MonthlyCharge",
    hue="Churn",
    ax=ax,
)

ax.set_title("Monthly Charge by Contract and Churn")
plt.tight_layout()
plt.show()


### 4.4 `sns.violinplot()` — quan sát hình dạng phân phối

Violin plot kết hợp ý tưởng của boxplot với **ước lượng mật độ**.

Cú pháp:

```python
sns.violinplot(
    data=df,
    x="Group",
    y="Value",
    hue="Class",
    split=True,
    inner="quart",
    ax=ax
)
```

Các tham số trong ví dụ:

- `split=True`: hai nhóm của `hue` được đặt hai nửa của cùng một violin.
- `inner="quart"`: hiển thị các quartile bên trong.
- `hue="Churn"`: so sánh churn và non-churn.

Violin plot hữu ích khi muốn nhìn không chỉ trung vị mà còn **hình dạng của phân phối**.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=customers,
    x="Contract",
    y="Tenure",
    hue="Churn",
    split=True,
    inner="quart",
    ax=ax,
)

ax.set_title("Tenure Distribution by Contract and Churn")
plt.tight_layout()
plt.show()

### Bài tập 3 — Categorical plots

1. Dùng `barplot()` để so sánh **Profit trung bình theo Category**, phân nhóm thêm theo `Region`.
2. Dùng `countplot()` để xem số khách hàng churn theo `Contract`.
3. Dùng `boxplot()` để so sánh `Tenure` giữa `Churn=0` và `Churn=1`.
4. Viết 2–3 nhận xét.

**Chú ý:** Khi dùng `barplot`, hãy nói rõ bạn đang hiển thị **mean**, không phải total.

In [ ]:
# TODO - Bài tập 3.1
fig, ax = plt.subplots(figsize=(10, 5))

# sns.barplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 3.2
fig, ax = plt.subplots(figsize=(7, 5))

# sns.countplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 3.3
fig, ax = plt.subplots(figsize=(7, 5))

# sns.boxplot(...)

plt.tight_layout()
plt.show()

## 5. Distribution plots — khám phá phân phối

Các hàm quan trọng:

- `histplot()` — histogram;
- `kdeplot()` — Kernel Density Estimate;
- `ecdfplot()` — empirical cumulative distribution.

Histogram cho thấy **số lượng/tần suất theo khoảng**.  
KDE cung cấp một đường ước lượng mượt của mật độ.


### 5.1 `sns.histplot()` — histogram

Histogram chia miền giá trị thành các khoảng (`bins`) và cho biết có bao nhiêu quan sát trong mỗi khoảng.

Cú pháp:

```python
sns.histplot(
    data=df,
    x="NumericColumn",
    bins=20,
    kde=True,
    ax=ax
)
```

Các tham số:

- `x=`: biến số cần xem phân phối.
- `bins=`: số khoảng.
- `kde=True`: vẽ thêm đường mật độ KDE.
- `color=`: màu histogram.

Histogram phù hợp để phát hiện:

- vùng dữ liệu tập trung;
- độ lệch;
- nhiều đỉnh;
- giá trị quá lớn/quá nhỏ.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    data=customers,
    x="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    bins=10,
    kde=True,
    ax=ax,
)

ax.set_title("Distribution of Monthly Charge")
plt.tight_layout()
plt.show()



### 5.2 Histogram theo nhóm với `hue`

Ta có thể dùng cùng `histplot()` nhưng thêm:

```python
hue="Churn"
```

để so sánh hai phân phối.

Ví dụ:

```python
sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=12,
    element="step",
    stat="density",
    common_norm=False
)
```

Ý nghĩa:

- `element="step"`: vẽ histogram dạng đường biên.
- `stat="density"`: chuẩn hóa theo mật độ thay vì count.
- `common_norm=False`: mỗi nhóm được chuẩn hóa riêng.
- `alpha=`: độ trong suốt.

Cách này hữu ích khi hai nhóm có số lượng quan sát khác nhau.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=10,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.35,
    ax=ax,
)

ax.set_title("Tenure Distribution by Churn")
plt.tight_layout()
plt.show()



### 5.3 `sns.kdeplot()` — Kernel Density Estimate

`kdeplot()` tạo một đường cong mượt để mô tả hình dạng phân phối.

Cú pháp:

```python
sns.kdeplot(
    data=df,
    x="NumericColumn",
    hue="Group",
    fill=True,
    common_norm=False,
    ax=ax
)
```

Các tham số:

- `hue=`: vẽ mật độ riêng cho từng nhóm.
- `fill=True`: tô vùng dưới đường KDE.
- `common_norm=False`: chuẩn hóa từng nhóm riêng.
- `alpha=`: độ trong suốt của vùng tô.

> KDE là một ước lượng mượt. Với dataset rất nhỏ, không nên diễn giải quá chi tiết từng gợn của đường cong.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.kdeplot(
    data=customers,
    x="MonthlyCharge",
    hue="Churn",
    fill=True,
    common_norm=False,
    alpha=0.35,
    ax=ax,
)

ax.set_title("Monthly Charge Density by Churn")
plt.tight_layout()
plt.show()

### Bài tập 4 — Distribution

1. Vẽ histogram `SupportCalls`, phân nhóm theo `Churn`.
2. Vẽ KDE cho `Tenure`, phân nhóm theo `Contract`.
3. So sánh histogram và KDE:
   - cái nào cho count rõ hơn?
   - cái nào cho shape rõ hơn?

In [ ]:
# TODO - Bài tập 4.1
fig, ax = plt.subplots(figsize=(8, 5))

# sns.histplot(...)

plt.tight_layout()
plt.show()

In [ ]:
# TODO - Bài tập 4.2
fig, ax = plt.subplots(figsize=(9, 5))

# sns.kdeplot(...)

plt.tight_layout()
plt.show()


## 6. Regression plots — quan hệ và đường hồi quy

### `sns.regplot()`

`regplot()` vẽ:

1. các điểm dữ liệu;
2. một đường hồi quy tuyến tính ước lượng.

Cú pháp:

```python
sns.regplot(
    data=df,
    x="X",
    y="Y",
    scatter_kws={"alpha": 0.5},
    line_kws={"linewidth": 2},
    ax=ax
)
```

- `scatter_kws`: tùy chỉnh scatter points.
- `line_kws`: tùy chỉnh regression line.
- `ax=`: có thể đặt vào subplot có sẵn.

`regplot()` là **axes-level function**.

> Đường hồi quy giúp mô tả xu hướng trong dữ liệu; nó không tự động chứng minh quan hệ nhân quả.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.regplot(
    data=retail,
    x="Revenue",
    y="Profit",
    color=sns.color_palette("deep")[0],
    scatter_kws={"alpha": 0.5},
    line_kws={"linewidth": 2},
    ax=ax,
)

ax.set_title("Regression View: Revenue vs Profit")
plt.tight_layout()
plt.show()


### `sns.lmplot()` — hồi quy kết hợp phân nhóm/facet

`lmplot()` cũng vẽ scatter + regression line nhưng là **figure-level function**.

Cú pháp:

```python
g = sns.lmplot(
    data=df,
    x="X",
    y="Y",
    hue="Group",
    height=5,
    aspect=1.4
)
```

Các tham số:

- `hue=`: tạo màu/đường hồi quy riêng cho từng nhóm.
- `height=`: chiều cao mỗi panel.
- `aspect=`: tỷ lệ chiều rộng / chiều cao.
- `scatter_kws=`: tùy chỉnh điểm.

Khác `regplot()`, `lmplot()` tự tạo Figure nên không dùng `ax=`.


In [ ]:
g = sns.lmplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    height=5,
    aspect=1.4,
    scatter_kws={"alpha": 0.5},
)

g.fig.suptitle("Revenue vs Profit by Category", y=1.03)
plt.show()

### Bài tập 5 — Regression plots

Dùng dữ liệu `economy`:

1. vẽ `Unemployment` vs `Inflation` bằng `regplot()`;
2. thêm title `"Inflation vs Unemployment"`;
3. tính correlation;
4. trả lời: có nên kết luận thất nghiệp gây ra lạm phát không?

In [ ]:
# TODO - Bài tập 5

corr = economy["Unemployment"].corr(economy["Inflation"])
print("Correlation:", round(corr, 3))

fig, ax = plt.subplots(figsize=(8, 5))

# sns.regplot(...)

plt.tight_layout()
plt.show()


## 7. Matrix plots — `sns.heatmap()`

Heatmap biểu diễn một **ma trận số** bằng màu.

Ứng dụng thường gặp:

- correlation matrix;
- pivot table;
- confusion matrix;
- bảng missing values.

### `DataFrame.corr()`

```python
corr = df[["x1", "x2", "x3"]].corr()
```

tạo ma trận hệ số tương quan giữa các biến số.

### `sns.heatmap()`

```python
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax
)
```

Các tham số:

- `annot=True`: ghi giá trị vào từng ô.
- `fmt=".2f"`: hiển thị 2 chữ số thập phân.
- `cmap=`: bảng màu.
- `center=0`: đặt giá trị 0 làm tâm của thang màu.

> Correlation mô tả mức độ liên hệ tuyến tính; correlation không đồng nghĩa với causation.


In [ ]:
corr = customers[
    ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
].corr()

fig, ax = plt.subplots(figsize=(7, 5))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
)

ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### Heatmap từ `pivot_table()`

Heatmap không chỉ dùng cho correlation.

Ta có thể chuyển dữ liệu dạng dài thành ma trận bằng:

```python
df.pivot_table(
    index="Region",
    columns="Category",
    values="Revenue",
    aggfunc="mean"
)
```

Trong đó:

- `index=`: tạo các hàng.
- `columns=`: tạo các cột.
- `values=`: biến được tổng hợp.
- `aggfunc="mean"`: phép tổng hợp.

Sau đó dùng `sns.heatmap()` để nhìn nhanh ô nào cao hoặc thấp.

Trong ví dụ dưới đây, ma trận thể hiện:

> **Average Revenue theo Region × Category**


In [ ]:
pivot_revenue = retail.pivot_table(
    index="Region",
    columns="Category",
    values="Revenue",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 5))

sns.heatmap(
    pivot_revenue,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=ax,
)

ax.set_title("Average Revenue: Region × Category")
plt.tight_layout()
plt.show()

### Bài tập 6 — Heatmap

1. Tạo correlation heatmap cho:
   - `Revenue`,
   - `Profit`,
   - `Orders`.
2. Tạo pivot table:
   - rows = `Category`
   - columns = `Region`
   - value = `Profit`
   - aggfunc = `"mean"`
3. Vẽ heatmap từ pivot table.
4. Region–Category nào có profit trung bình cao nhất?

In [ ]:
# TODO - Bài tập 6.1

In [ ]:
# TODO - Bài tập 6.2


## 8. `sns.pairplot()` — khám phá nhiều biến cùng lúc

`pairplot()` tự tạo nhiều biểu đồ để xem quan hệ giữa các cặp biến.

Cú pháp:

```python
sns.pairplot(
    df[["x1", "x2", "x3", "class"]],
    hue="class",
    corner=True
)
```

Nó thường tạo:

- scatter plot cho từng cặp biến;
- distribution plot trên đường chéo.

Các tham số:

- `hue=`: tô màu theo nhóm.
- `corner=True`: chỉ vẽ nửa dưới của ma trận, tránh lặp lại biểu đồ đối xứng.

### `DataFrame.sample()`

Nếu dataset lớn, có thể lấy mẫu:

```python
sample_df = df.sample(400, random_state=42)
```

Trong notebook này dataset nhỏ, nên ta có thể dùng toàn bộ dữ liệu.

`pairplot()` rất hữu ích trong **EDA ban đầu**, nhưng với quá nhiều biến thì Figure sẽ rất lớn.


In [ ]:

# Dataset mẫu nhỏ nên dùng toàn bộ khách hàng
sample_customers = customers.copy()

g = sns.pairplot(
    sample_customers[
        ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
    ],
    hue="Churn",
    corner=True,
)

g.fig.suptitle("Pairwise EDA for Customer Churn", y=1.02)
plt.show()


### Bài tập 7 — Pairplot

Tạo `pairplot()` cho Retail với:

- `Revenue`
- `Profit`
- `Orders`
- `Category`

Yêu cầu:

- `hue="Category"`
- `corner=True`

Sau đó tìm ít nhất **2 pattern** có thể quan sát được.

In [ ]:
# TODO - Bài tập 7

## 9. Axes-level và Figure-level APIs

### Axes-level

Ví dụ:

- `scatterplot`
- `lineplot`
- `barplot`
- `boxplot`
- `histplot`
- `regplot`
- `heatmap`

Có thể truyền `ax=`.

### Figure-level

Ví dụ:

- `relplot`
- `catplot`
- `displot`
- `lmplot`

Chúng tự quản lý Figure và hỗ trợ faceting bằng `row=` / `col=`.


### `sns.relplot()` — relational plot nhiều panel

`relplot()` là phiên bản **figure-level** cho relational plots.

Cú pháp:

```python
g = sns.relplot(
    data=df,
    x="X",
    y="Y",
    hue="Group",
    col="FacetVariable",
    col_wrap=2,
    height=4
)
```

Các tham số:

- `hue=`: phân nhóm bằng màu.
- `col=`: tạo một panel cho mỗi giá trị của biến.
- `col_wrap=2`: sau 2 panel thì xuống dòng.
- `height=`: chiều cao mỗi panel.

Mặc định `kind="scatter"`.


In [ ]:
g = sns.relplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    col="Region",
    col_wrap=2,
    height=4,
)

g.fig.suptitle("Revenue vs Profit by Region", y=1.02)
plt.show()


### `sns.catplot()` — categorical plot nhiều panel

`catplot()` là **figure-level function** cho categorical plots.

Cú pháp:

```python
sns.catplot(
    data=df,
    x="Group",
    y="Value",
    col="Class",
    kind="box"
)
```

`kind=` quyết định loại biểu đồ, ví dụ:

```text
bar, count, box, violin, strip, swarm
```

Trong ví dụ dưới đây:

- `kind="box"` → boxplot;
- `col="Churn"` → một panel cho mỗi trạng thái churn.


In [ ]:
g = sns.catplot(
    data=customers,
    x="Contract",
    y="Tenure",
    col="Churn",
    kind="box",
    color=sns.color_palette("deep")[0],
    height=4,
    aspect=1.1,
)

g.fig.suptitle("Tenure by Contract, Faceted by Churn", y=1.03)
plt.show()


### `sns.displot()` — distribution plot nhiều panel

`displot()` là **figure-level function** cho phân phối.

Cú pháp:

```python
sns.displot(
    data=df,
    x="Value",
    col="Group",
    hue="Class",
    kind="hist"
)
```

Các giá trị `kind` thường gặp:

```text
hist, kde, ecdf
```

Trong ví dụ:

- `col="Contract"` tạo một panel cho từng loại hợp đồng;
- `hue="Churn"` so sánh churn trong mỗi panel;
- `kind="hist"` dùng histogram.


In [ ]:
g = sns.displot(
    data=customers,
    x="MonthlyCharge",
    col="Contract",
    hue="Churn",
    kind="hist",
    bins=8,
    common_norm=False,
    height=4,
)

g.fig.suptitle("Monthly Charge Distribution by Contract", y=1.03)
plt.show()


### Bài tập 8 — Figure-level plots

1. Dùng `relplot()` để vẽ Revenue–Profit:
   - `hue="Category"`
   - `col="Region"`
2. Dùng `catplot(kind="box")` để vẽ Tenure theo Contract, `col="Churn"`.
3. Giải thích khi nào Figure-level thuận tiện hơn Axes-level.

In [ ]:
# TODO - Bài tập 8.1

In [ ]:
# TODO - Bài tập 8.2


## 10. Theme, style, context và palette

### `sns.set_theme()`

Thiết lập đồng thời style và context:

```python
sns.set_theme(style="ticks", context="talk")
```

### `sns.set_style()`

Chỉ thay đổi nền và grid:

```python
sns.set_style("whitegrid")
```

### `sns.set_context()`

Điều chỉnh kích thước chữ, đường và marker:

```python
sns.set_context("talk")
```

### `sns.color_palette()`

Lấy một bảng màu:

```python
sns.color_palette("deep")
```

### `sns.despine()`

Loại bỏ các đường viền trên/phải:

```python
sns.despine()
```

Một số style:

```text
whitegrid, darkgrid, white, dark, ticks
```

Một số context:

```text
paper, notebook, talk, poster
```

Theme giúp biểu đồ nhất quán, nhưng không thay thế việc chọn đúng loại biểu đồ và ghi nhãn rõ ràng.


In [ ]:
sns.set_theme(style="ticks", context="talk")

fig, ax = plt.subplots(figsize=(9, 5))

sns.lineplot(
    data=monthly_region,
    x="Month",
    y="Revenue",
    hue="Region",
    marker="o",
    ax=ax,
)

sns.despine()
ax.set_title("Same Data, Different Seaborn Theme")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# Trả về mặc định cho notebook
sns.set_theme(style="whitegrid", context="notebook")

### Bài tập 9 — Style

Tạo cùng một biểu đồ dưới 2 style khác nhau:

- `whitegrid`
- `ticks`

Viết nhận xét:

- style nào phù hợp báo cáo Business?
- style nào phù hợp slide thuyết trình?
- vì sao đây là quyết định thiết kế chứ không phải quyết định thống kê?

In [ ]:
# TODO - Bài tập 9


## 11. Subplots — kết hợp nhiều biểu đồ Seaborn

Seaborn axes-level functions có thể được đặt vào dashboard bằng `plt.subplots()`.

Ví dụ:

```python
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(..., ax=axes[0])
sns.scatterplot(..., ax=axes[1])
sns.histplot(..., ax=axes[2])
```

### Khi `axes` có nhiều phần tử

- `axes[0]`, `axes[1]`, ... với layout 1 hàng.
- `axes[0, 0]`, `axes[0, 1]`, ... với layout 2 chiều.

### `fig.suptitle()`

Thêm tiêu đề chung cho toàn dashboard:

```python
fig.suptitle("Dashboard")
```

Đây là một lý do quan trọng để hiểu mối quan hệ giữa Seaborn và Matplotlib:
**Seaborn vẽ nội dung; Matplotlib quản lý Figure và layout.**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

sns.barplot(
    data=retail,
    x="Region",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0],
)
axes[0].set_title("Average Revenue")

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Revenue vs Profit")

sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=10,
    element="step",
    ax=axes[2],
)
axes[2].set_title("Customer Tenure")

fig.suptitle("Mini Analytics Dashboard", fontsize=16, y=1.03)
plt.tight_layout()
plt.show()


### Bài tập 10 — Dashboard 2×2

Tạo dashboard gồm:

1. `lineplot`: Revenue theo Month;
2. `barplot`: Profit theo Category;
3. `boxplot`: MonthlyCharge theo Churn;
4. `heatmap`: correlation của customer numerical variables.

Yêu cầu:

- `figsize=(14, 9)`
- title cho từng biểu đồ;
- `fig.suptitle(...)`;
- `plt.tight_layout()`.

In [ ]:
# TODO - Bài tập 10

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# TODO: axes[0, 0]
# TODO: axes[0, 1]
# TODO: axes[1, 0]
# TODO: axes[1, 1]

fig.suptitle("Seaborn Analytics Dashboard", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


## 12. Annotation — khi Seaborn cần Matplotlib

Seaborn giúp tạo plot nhanh, còn việc nhấn mạnh thông tin thường dùng Matplotlib.

### `ax.axhline()`

Vẽ đường ngang tham chiếu:

```python
ax.axhline(0, linestyle="--")
```

Ví dụ: đường `GDP Growth = 0`.

### `ax.axvline()`

Vẽ đường thẳng đứng tại một giá trị X.

### `ax.axvspan()`

Tô một khoảng trên trục X.

### `ax.annotate()`

Ghi chú trực tiếp lên biểu đồ:

```python
ax.annotate(
    "Lowest GDP Growth",
    xy=(x, y),
    xytext=(x_text, y_text),
    arrowprops={"arrowstyle": "->"}
)
```

- `xy=`: vị trí điểm cần chú thích.
- `xytext=`: vị trí của đoạn text.
- `arrowprops=`: thiết lập mũi tên.

Ví dụ dưới đây đánh dấu quý có GDP Growth thấp nhất.


In [ ]:
min_idx = economy["GDP_Growth"].idxmin()
x_min = min_idx
y_min = economy.loc[min_idx, "GDP_Growth"]

fig, ax = plt.subplots(figsize=(11, 5))

sns.lineplot(
    data=economy,
    x="Quarter",
    y="GDP_Growth",
    color=sns.color_palette("deep")[0],
    marker="o",
    ax=ax,
)

ax.axhline(0, linestyle="--", linewidth=1)

ax.annotate(
    f'Min: {economy.loc[min_idx, "Quarter"]}\n{y_min:.2f}',
    xy=(x_min, y_min),
    xytext=(x_min - 4, y_min + 1.1),
    arrowprops={"arrowstyle": "->"},
)

ax.set_title("GDP Growth with Annotation")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 11 — Data storytelling

Trên lineplot tổng Revenue theo Month:

1. tìm tháng Revenue cao nhất;
2. dùng `annotate()` đánh dấu;
3. thêm đường mean bằng `ax.axhline()`;
4. đổi title thành một **message title**, không chỉ là tên biểu đồ.

Ví dụ:

> `"Revenue Peaks at Year-End After a Mid-Year Slowdown"`

In [ ]:
# TODO - Bài tập 11


## 13. Ứng dụng Data Science — EDA cho Customer Churn

Một workflow EDA trực quan cơ bản:

1. **Target distribution** → `countplot()`
2. **Numerical vs target** → `boxplot()`
3. **Correlation** → `.corr()` + `heatmap()`
4. **Multivariate relationships** → `scatterplot()` / `pairplot()`

Ở phần này không có hàm Seaborn mới. Ta **kết hợp lại các hàm đã học** trên cùng một dashboard.

Mục tiêu là trả lời:

- tỷ lệ churn có cân bằng không;
- tenure của hai nhóm có khác nhau không;
- monthly charge có khác nhau không;
- các biến numerical liên hệ với nhau ra sao.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.countplot(
    data=customers,
    x="Churn",
    color=sns.color_palette("deep")[0],
    ax=axes[0, 0],
)
axes[0, 0].set_title("Target Distribution")

sns.boxplot(
    data=customers,
    x="Churn",
    y="Tenure",
    color=sns.color_palette("deep")[0],
    ax=axes[0, 1],
)
axes[0, 1].set_title("Tenure vs Churn")

sns.boxplot(
    data=customers,
    x="Churn",
    y="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    ax=axes[1, 0],
)
axes[1, 0].set_title("Monthly Charge vs Churn")

customer_corr = customers[
    ["Tenure", "MonthlyCharge", "SupportCalls", "Churn"]
].corr()

sns.heatmap(
    customer_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Correlation Matrix")

fig.suptitle("Customer Churn — EDA Overview", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Câu hỏi Data Science

Dựa trên các biểu đồ:

1. Target có mất cân bằng nghiêm trọng không?
2. `Tenure` có vẻ hữu ích cho dự đoán churn không?
3. `MonthlyCharge` có khác giữa hai nhóm không?
4. `SupportCalls` có tương quan với churn không?
5. Có thể chọn feature chỉ dựa vào correlation không?

Viết câu trả lời bằng ngôn ngữ phân tích, tránh khẳng định nhân quả.


## 14. Ứng dụng Business — Sales & Profitability

Mục tiêu là tạo một dashboard hỗ trợ manager.

Ta kết hợp:

- `lineplot()` → xu hướng doanh thu;
- `barplot()` → so sánh lợi nhuận giữa các khu vực;
- `scatterplot()` → Revenue–Profit;
- `heatmap()` → Profit theo Region × Category.

Trước khi vẽ, ta dùng Pandas:

```python
groupby(...).sum()
pivot_table(...)
```

để đưa dữ liệu về đúng cấu trúc cho từng biểu đồ.

> Một dashboard tốt không chỉ chứa nhiều biểu đồ; mỗi biểu đồ nên trả lời một câu hỏi quản trị cụ thể.


In [ ]:
monthly_total = retail.groupby("Month", as_index=False)["Revenue"].sum()
region_profit = retail.groupby("Region", as_index=False)["Profit"].sum()
profit_matrix = retail.pivot_table(
    index="Region",
    columns="Category",
    values="Profit",
    aggfunc="sum",
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.lineplot(
    data=monthly_total,
    x="Month",
    y="Revenue",
    color=sns.color_palette("deep")[0],
    marker="o",
    ax=axes[0, 0],
)
axes[0, 0].set_title("Monthly Revenue")
axes[0, 0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=region_profit,
    x="Region",
    y="Profit",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Total Profit by Region")

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    size="Orders",
    sizes=(20, 180),
    alpha=0.6,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Revenue vs Profit")

sns.heatmap(
    profit_matrix,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=axes[1, 1],
)
axes[1, 1].set_title("Profit: Region × Category")

fig.suptitle("Retail Business Performance", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Câu hỏi Business

Viết **3–5 insights** theo cấu trúc:

- **Observation**
- **Evidence**
- **Business implication**

Ví dụ:

> Online có doanh thu cao nhưng cần xem đồng thời profit để tránh kết luận rằng doanh thu cao đồng nghĩa hiệu quả cao.


## 15. Ứng dụng Economics — Inflation, Unemployment, GDP Growth

Ở phần này ta sử dụng `lineplot()` nhiều lần trên các `Axes` khác nhau để đặt ba chỉ số
trên cùng một trục thời gian.

Ta cũng dùng:

```python
axes[2].axhline(0, ...)
```

để thêm đường tham chiếu tại `GDP Growth = 0`.

Seaborn hỗ trợ tốt:

- vẽ time series;
- so sánh các macro indicators;
- xem scatter/regression relationship.

Tuy nhiên, trực quan hóa chỉ giúp **mô tả và khám phá** dữ liệu; nó không thay thế mô hình kinh tế lượng khi cần suy luận nhân quả.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

sns.lineplot(
    data=economy,
    x="Quarter",
    y="Inflation",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Inflation")

sns.lineplot(
    data=economy,
    x="Quarter",
    y="Unemployment",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Unemployment")

sns.lineplot(
    data=economy,
    x="Quarter",
    y="GDP_Growth",
    marker="o",
    ax=axes[2],
)
axes[2].axhline(0, linestyle="--", linewidth=1)
axes[2].set_title("GDP Growth")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Bài tập 12 — Economics visualization

Tạo Figure gồm 2 biểu đồ:

**Trái**
- `regplot`: Unemployment vs Inflation

**Phải**
- `scatterplot`: GDP Growth vs Inflation
- dùng `size="Unemployment"` bằng cách truyền column tương ứng

Sau đó viết 4–6 câu nhận xét và nêu rõ:

> Correlation/visual association ≠ causation.

In [ ]:
# TODO - Bài tập 12
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# TODO: regplot bên trái
# TODO: scatterplot bên phải

plt.tight_layout()
plt.show()


## 16. Xuất biểu đồ

Dù plot được tạo bởi Seaborn, Figure vẫn là Matplotlib Figure.

### `Path.mkdir()`

```python
output_dir.mkdir(exist_ok=True)
```

tạo thư mục nếu chưa tồn tại.

### `fig.savefig()`

```python
fig.savefig(
    "chart.png",
    dpi=180,
    bbox_inches="tight"
)
```

Các tham số:

- `dpi=`: độ phân giải ảnh.
- `bbox_inches="tight"`: cắt bớt khoảng trắng thừa.

Có thể lưu thành:

```text
.png, .jpg, .pdf, .svg
```

Ví dụ dưới đây lưu scatter plot Revenue–Profit thành PNG.


In [ ]:
output_dir = Path("seaborn_outputs")
output_dir.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=retail,
    x="Revenue",
    y="Profit",
    hue="Category",
    ax=ax,
)

ax.set_title("Revenue vs Profit")

output_file = output_dir / "revenue_profit.png"
fig.savefig(output_file, dpi=180, bbox_inches="tight")

plt.show()

print("Saved to:", output_file.resolve())


## 17. Viết hàm vẽ tái sử dụng

Khi phải tạo cùng một loại biểu đồ nhiều lần, ta nên đóng gói code thành hàm.

Ví dụ:

```python
def plot_group_boxplot(data, x, y, ax=None, title=None):
    ...
```

Ý nghĩa các tham số:

- `data`: DataFrame.
- `x`: tên biến categorical.
- `y`: tên biến numerical.
- `ax=None`: cho phép truyền Axes từ bên ngoài.
- `title=None`: tiêu đề tùy chọn.

### Vì sao nhận `ax`?

Nếu hàm luôn tự tạo Figure, ta khó đưa nó vào dashboard.

Mẫu tốt:

```python
if ax is None:
    fig, ax = plt.subplots(...)
```

Nghĩa là:

- nếu người gọi truyền `ax` → vẽ vào dashboard có sẵn;
- nếu không truyền → hàm tự tạo Axes mới.

Cuối hàm trả về:

```python
return ax
```

để người dùng có thể tiếp tục tùy chỉnh biểu đồ.


In [ ]:
def plot_group_boxplot(data, x, y, ax=None, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))

    sns.boxplot(
        data=data,
        x=x,
        y=y,
        color=sns.color_palette("deep")[0],
        ax=ax,
    )

    ax.set_title(title or f"{y} by {x}")
    return ax


fig, ax = plt.subplots(figsize=(8, 5))
plot_group_boxplot(
    customers,
    x="Contract",
    y="MonthlyCharge",
    ax=ax,
    title="Monthly Charge by Contract",
)
plt.tight_layout()
plt.show()

### Bài tập 13 — Mở rộng hàm

Sửa hàm trên để nhận thêm:

- `hue=None`
- `palette=None`

Sau đó dùng hàm để vẽ:

- `Tenure` theo `Contract`
- `hue="Churn"`

In [ ]:
# TODO - Bài tập 13

## 18. Bài thực hành có lời giải — EDA nhanh cho Customer Churn

### Đề bài

Trong vai Data Analyst, hãy tạo một báo cáo trực quan ngắn để trả lời:

1. Churn rate khoảng bao nhiêu?
2. Contract type nào có churn rate cao hơn?
3. Khách hàng churn có tenure khác không?
4. Monthly charge có liên hệ với churn không?
5. Feature nào nên được xem xét ở bước modeling?

Phần dưới là **một lời giải mẫu**.

In [ ]:
# 1. Tạo churn rate theo contract
churn_by_contract = (
    customers.groupby("Contract", as_index=False)["Churn"]
    .mean()
    .sort_values("Churn", ascending=False)
)

display(churn_by_contract)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) Churn rate theo contract
sns.barplot(
    data=churn_by_contract,
    x="Contract",
    y="Churn",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Churn Rate by Contract")
axes[0, 0].set_ylabel("Churn Rate")

# (2) Tenure distribution
sns.histplot(
    data=customers,
    x="Tenure",
    hue="Churn",
    bins=10,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Tenure Distribution")

# (3) Monthly Charge
sns.boxplot(
    data=customers,
    x="Churn",
    y="MonthlyCharge",
    color=sns.color_palette("deep")[0],
    ax=axes[1, 0],
)
axes[1, 0].set_title("Monthly Charge by Churn")

# (4) Multivariate view
# Dataset nhỏ nên sử dụng toàn bộ khách hàng.
sns.scatterplot(
    data=customers,
    x="Tenure",
    y="MonthlyCharge",
    hue="Churn",
    size="SupportCalls",
    sizes=(20, 150),
    alpha=0.65,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Customer Profile")

fig.suptitle("Customer Churn — Guided EDA", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


### Gợi ý diễn giải lời giải

Một báo cáo tốt không chỉ mô tả biểu đồ mà phải kết nối với câu hỏi phân tích.

Ví dụ:

- Nếu nhóm Monthly contract có churn rate cao hơn, `Contract` là biến đáng kiểm tra trong modeling.
- Nếu churn tập trung ở tenure thấp, doanh nghiệp nên chú ý giai đoạn onboarding/early lifecycle.
- Nếu MonthlyCharge của nhóm churn cao hơn, giá có thể là một tín hiệu liên quan — nhưng chưa thể kết luận là nguyên nhân.
- `SupportCalls` có thể phản ánh friction/service issues và nên được kiểm tra thêm.
- Cần tiếp tục đánh giá bằng mô hình, validation và domain knowledge.

## 19. Bài tập tổng hợp tự làm

### PROJECT — Data Visualization Report bằng Seaborn

Chọn **một trong ba bối cảnh**:

### A. Business Analytics
Dùng `retail`.

### B. Data Science
Dùng `customers`.

### C. Economics
Dùng `economy`.

### Sản phẩm cần nộp

Một notebook/report gồm ít nhất:

1. **1 relational plot**
2. **1 categorical plot**
3. **1 distribution plot**
4. **1 heatmap hoặc pairplot**
5. **1 dashboard ít nhất 2×2**
6. **ít nhất 1 annotation**
7. **5 insights**
8. **2 đề xuất ra quyết định / bước phân tích tiếp theo**

### Ràng buộc

- Mỗi hình phải có title và labels phù hợp.
- Không dùng biểu đồ chỉ vì “đẹp”.
- Phải giải thích tại sao chọn loại biểu đồ đó.
- Không suy diễn correlation thành causation.

### Khung làm bài

#### Bước 1 — Câu hỏi phân tích

Viết 3–5 câu hỏi trước khi vẽ.

> 1. ...
>
> 2. ...
>
> 3. ...

#### Bước 2 — Univariate analysis

> ...

#### Bước 3 — Bivariate / multivariate analysis

> ...

#### Bước 4 — Dashboard

> ...

#### Bước 5 — Insights

> 1. ...
>
> 2. ...
>
> 3. ...
>
> 4. ...
>
> 5. ...

#### Bước 6 — Recommendation / next steps

> ...

In [ ]:
# TODO - PROJECT TỰ LÀM

# Chọn dataset:
# df = retail.copy()
# df = customers.copy()
# df = economy.copy()

# Bắt đầu phân tích tại đây.

## 20. Cheat sheet lệnh Seaborn

| Mục đích | Hàm |
|---|---|
| Scatter | `sns.scatterplot()` |
| Line | `sns.lineplot()` |
| Bar mean/estimator | `sns.barplot()` |
| Count | `sns.countplot()` |
| Boxplot | `sns.boxplot()` |
| Violin | `sns.violinplot()` |
| Histogram | `sns.histplot()` |
| KDE | `sns.kdeplot()` |
| ECDF | `sns.ecdfplot()` |
| Regression | `sns.regplot()` |
| Regression + facet | `sns.lmplot()` |
| Heatmap | `sns.heatmap()` |
| Pairwise EDA | `sns.pairplot()` |
| Relational facet | `sns.relplot()` |
| Categorical facet | `sns.catplot()` |
| Distribution facet | `sns.displot()` |
| Theme | `sns.set_theme()` |
| Style | `sns.set_style()` |
| Context | `sns.set_context()` |
| Remove spines | `sns.despine()` |

### Semantic mappings

| Tham số | Ý nghĩa |
|---|---|
| `x` | biến trục X |
| `y` | biến trục Y |
| `hue` | phân nhóm bằng màu |
| `style` | phân nhóm bằng marker/line style |
| `size` | phân nhóm bằng kích thước |
| `row` | facet theo hàng |
| `col` | facet theo cột |
| `data` | DataFrame |
| `ax` | Matplotlib Axes |

## Kết thúc

Sau notebook này, người học nên phân biệt được:

### Khi nào dùng Matplotlib?
- cần kiểm soát layout/annotation chi tiết;
- cần custom graphics;
- cần xây Figure phức tạp.

### Khi nào dùng Seaborn?
- làm việc với DataFrame;
- cần statistical visualization nhanh;
- cần `hue`, `style`, `size`, faceting;
- cần EDA hiệu quả với ít code.

### Thực tế
Hai thư viện thường được **dùng cùng nhau**:

```python
fig, ax = plt.subplots()
sns.someplot(..., ax=ax)
ax.set_title(...)
ax.annotate(...)
plt.show()
```

Mục tiêu cuối cùng không phải “biết nhiều hàm vẽ”, mà là:

> **chọn đúng biểu đồ → đọc đúng dữ liệu → truyền đạt đúng insight.**